# 🏭 Titan Maintenance Intelligence Platform (MIP)
### Agentic AI — Final Assignment · Challenge 1: Asset Reliability & Predictive Maintenance

**Client:** Titan Manufacturing Corporation  
**Architecture:** Hierarchical Multi-Agent (ReAct) — *Observe → Reason → Act → Learn*  
**Framework:** CrewAI (same as the course labs)  
**LLM:** Gemini free tier via Google AI Studio (same as the course labs)

This notebook builds the complete **5-agent system** from our *Agent Prompts & Tool Descriptions* deliverable:

| # | Agent | Role |
|---|-------|------|
| 1 | Maintenance Intelligence Orchestrator | Manager / coordinator |
| 2 | Reliability Intelligence Agent | Specialist — asset health & failure risk |
| 3 | Process Intelligence Agent | Specialist — workflow & bottlenecks |
| 4 | Cost Intelligence Agent | Specialist — financial impact |
| 5 | Safety & Governance Agent | Governance gate (final approval) |

> Run the cells **top to bottom**. The only thing you must edit is your Gemini API key in Cell 2.


## Cell 1 — Install CrewAI
Run this first. Wait for the green checkmark before moving on. Ignore the dependency warnings.

In [ ]:
!pip install crewai crewai_tools -q

## Cell 2 — Configure your API key
Get a free Gemini key at **https://aistudio.google.com → "Get API Key"**, then paste it below (keep the quotes).

In [ ]:
import os

# Replace with YOUR actual Gemini key (keep the quotes).
os.environ["GEMINI_API_KEY"] = "Paste your API KEY HERE"

# Tells CrewAI which model every agent should use by default.
os.environ["MODEL"] = "gemini/gemini-flash-lite-latest"

print("API key configured!")

## Cell 3 — Tool layer
> *"Agents decide. Tools execute."* The agent never touches a database or model directly — it calls one of these tools.

All **12 tools** from the deliverable are below. They are **mocked with realistic data** (no live SCADA / Helix in the lab), so the crew runs end-to-end and the *CNC-Lathe-07* walkthrough reproduces the numbers from the design document. Swap the bodies for real API calls when connecting to Titan's systems.

In [ ]:
import json
from crewai.tools import tool


# --- READ tools -------------------------------------------------------------

@tool("sensor_telemetry_api")
def sensor_telemetry_api(asset_id: str, window: str = "72h") -> str:
    """Fetches real-time and historical sensor readings from SCADA/historian
    for a given asset ID and time window. Returns vibration, temperature,
    pressure and current trends plus a data-freshness flag."""
    telemetry = {
        "asset_id": asset_id,
        "window": window,
        "spindle_vibration_g": {"current": 0.48, "baseline": 0.22, "trend": "rising since 04:00"},
        "temperature_c": {"current": 78, "baseline": 66, "pct_above_baseline": 18},
        "pressure_bar": {"current": 5.1, "baseline": 5.0},
        "motor_current_a": {"current": 14.7, "baseline": 12.0},
        "data_freshness_minutes": 2,          # < 15 min => data is fresh
        "data_quality": "OK",
    }
    return json.dumps(telemetry)


@tool("maintenance_history_api")
def maintenance_history_api(asset_id: str, last_n_tickets: int = 10) -> str:
    """Retrieves past tickets, repairs and recurring fault patterns for a given
    asset. Feeds the agent's episodic memory."""
    history = {
        "asset_id": asset_id,
        "tickets_reviewed": last_n_tickets,
        "recurring_pattern": "2 spindle bearing replacements in the last 90 days",
        "avg_resolution_time_hours": 26,
        "repeated_process_delay": "parts procurement delayed on 2 of the last 3 tickets",
        "last_tickets": [
            {"id": "TIT-2026-3990", "fault": "spindle bearing wear", "resolution_h": 31},
            {"id": "TIT-2026-3712", "fault": "spindle bearing wear", "resolution_h": 22},
        ],
    }
    return json.dumps(history)


@tool("downtime_cost_calculator")
def downtime_cost_calculator(asset_id: str) -> str:
    """Returns the financial impact ($/day) of an asset going offline,
    cross-referenced against the live production schedule."""
    cost = {
        "asset_id": asset_id,
        "daily_downtime_cost_usd": 180000,            # critical CNC machine
        "production_orders_due_14d": 3,               # 3 orders due Friday
        "orders_depend_on_this_asset": True,
    }
    return json.dumps(cost)


@tool("parts_inventory_api")
def parts_inventory_api(asset_id: str, fault_type: str = "spindle bearing") -> str:
    """Checks spare parts availability for a given asset and fault type. Returns
    on-hand count and lead time if unavailable."""
    inventory = {
        "asset_id": asset_id,
        "fault_type": fault_type,
        "part_number": "SKF bearing 6208",
        "on_hand": 1,
        "available": True,
        "lead_time_hours": 0,                # in stock; if 0 on_hand, would be 96h
        "parts_cost_usd": 240,
    }
    return json.dumps(inventory)


# --- EXECUTE tools ----------------------------------------------------------

@tool("anomaly_detector")
def anomaly_detector(asset_id: str, telemetry_stream: str = "") -> str:
    """Runs ML anomaly scoring on a telemetry stream. Returns anomaly flag,
    severity score (0-1) and contributing signals."""
    result = {
        "asset_id": asset_id,
        "anomaly_detected": True,
        "anomaly_severity": 0.82,
        "contributing_signals": ["spindle_vibration", "temperature", "motor_current"],
    }
    return json.dumps(result)


@tool("rul_predictor")
def rul_predictor(asset_id: str, degradation_data: str = "") -> str:
    """Estimates Remaining Useful Life (RUL) in hours for an asset based on
    degradation curves and historical failure patterns."""
    result = {"asset_id": asset_id, "rul_hours": 58, "confidence": 0.79}
    return json.dumps(result)


@tool("asset_health_scorer")
def asset_health_scorer(asset_id: str, anomaly_score: float = 0.0,
                        rul_hours: float = 0.0, history_summary: str = "") -> str:
    """Aggregates telemetry, anomaly score and maintenance history into a single
    asset health score (0-100) and a 7-day failure probability."""
    result = {
        "asset_id": asset_id,
        "health_score": 31,                 # 0 = failed, 100 = healthy
        "failure_probability_7d": 0.87,
    }
    return json.dumps(result)


@tool("root_cause_analyzer")
def root_cause_analyzer(asset_id: str, maintenance_tickets: str = "",
                        telemetry_summary: str = "") -> str:
    """Runs process-mining logic across maintenance tickets and telemetry to
    surface the likely root cause and contributing process-level factors."""
    result = {
        "asset_id": asset_id,
        "likely_root_cause": "spindle bearing fatigue (consistent with prior history)",
        "process_factor": "recurring delay in spare-parts procurement on this asset",
    }
    return json.dumps(result)


@tool("notification_service")
def notification_service(recipient: str, message: str) -> str:
    """Sends an alert to the relevant team (maintenance lead, plant manager) via
    email or the Helix portal with a risk summary and recommended next step."""
    return json.dumps({"status": "SENT", "recipient": recipient, "message": message})


@tool("policy_validator")
def policy_validator(action_type: str, asset_id: str, proposed_action: str = "") -> str:
    """Checks a proposed action against safety rules, LOTO procedures and
    financial approval thresholds. Returns APPROVED / REQUIRES_APPROVAL /
    REJECTED and gates all critical actions."""
    action = (action_type or "").lower() + " " + (proposed_action or "").lower()
    if "loto" in action or "override" in action:
        decision, approver = "REJECTED", None
    elif "shutdown" in action or "offline" in action:
        decision, approver = "REQUIRES_APPROVAL", "Plant Manager"
    elif "repair" in action and ">10000" in action.replace(" ", ""):
        decision, approver = "REQUIRES_APPROVAL", "Finance"
    elif any(k in action for k in ["alert", "notify", "work order", "dashboard", "draft"]):
        decision, approver = "APPROVED", None
    else:
        decision, approver = "REQUIRES_APPROVAL", "Maintenance Lead"
    return json.dumps({"decision": decision, "approver_required": approver,
                       "action_type": action_type, "asset_id": asset_id})


# --- WRITE tools (require human review) -------------------------------------

@tool("work_order_generator")
def work_order_generator(asset_id: str, fault: str, recommended_action: str,
                         priority: str, parts_needed: str = "",
                         technician: str = "") -> str:
    """Drafts a structured maintenance work order (asset, fault, recommended
    action, priority, parts needed, technician assignment) for human review."""
    work_order = {
        "work_order_id": "TIT-2026-4471",
        "asset_id": asset_id,
        "fault": fault,
        "recommended_action": recommended_action,
        "priority": priority,
        "parts_needed": parts_needed,
        "technician": technician,
        "status": "DRAFT — awaiting maintenance lead review",
    }
    return json.dumps(work_order)


@tool("dashboard_updater")
def dashboard_updater(asset_id: str, risk_assessment: str,
                      recommended_action: str) -> str:
    """Writes the current risk assessment and recommended action to the Helix
    maintenance dashboard for technician visibility."""
    return json.dumps({"status": "DASHBOARD_UPDATED", "asset_id": asset_id})

## Cell 4 — Define the agents
Each agent is a faithful CrewAI translation of its **system prompt** from the *Agent Prompts* deliverable: the full prompt lives in `backstory` (where CrewAI injects detailed behaviour and rules), `goal` is the one-line mission, and each specialist is handed exactly the tools its prompt is allowed to call.

In [ ]:
from crewai import Agent, Task, Crew, Process


# 1. ORCHESTRATOR — manager of the crew (coordinates, never analyses itself)
orchestrator = Agent(
    role="Maintenance Intelligence Orchestrator",
    goal=(
        "Coordinate the specialist agents to turn a maintenance event into a "
        "single prioritized, predictive and actionable recommendation, and "
        "decide whether to act autonomously or escalate to a human."
    ),
    backstory=(
        "You are the Maintenance Intelligence Orchestrator for Titan "
        "Manufacturing Corporation. You coordinate a team of specialist AI "
        "agents to convert fragmented machine telemetry and maintenance data "
        "into prioritized, predictive, actionable maintenance decisions. You do "
        "not perform analysis yourself — you direct the Reliability, Process and "
        "Cost specialists, then request a Safety & Governance review, and "
        "synthesize their findings into one recommendation.\n\n"
        "On every event you: (1) extract asset ID, plant, alert type, severity, "
        "timestamp; (2) gather assessments from the Reliability, Process and "
        "Cost agents; (3) request a Safety & Governance review of the proposed "
        "action; (4) CHECK FOR CONFLICTS between specialist outputs before "
        "scoring — e.g. Cost recommends deferring while Reliability flags "
        "urgency, or Process flags a bottleneck that contradicts the timeline. "
        "If a conflict exists, state it explicitly; never resolve it silently "
        "inside the formula. (5) Synthesize using the priority formula:\n"
        "    Priority = (failure_probability x 0.4) "
        "+ (financial_impact_score x 0.35) + (process_risk x 0.25)\n"
        "If Safety & Governance flags a risk the formula does not capture, "
        "Safety OVERRIDES the numeric priority — say so explicitly. "
        "(6) Decide AUTONOMOUS only if Priority < 0.7 AND the action does not "
        "require machine shutdown, repair approval > $10,000, or a safety "
        "override; otherwise ESCALATE.\n\n"
        "HARD CONSTRAINTS: NEVER recommend taking a CNC machine offline without "
        "human approval. NEVER approve repairs exceeding $10,000 autonomously. "
        "NEVER let the priority formula average away a disagreement — surface it "
        "in conflicts_detected first. ALWAYS give a rationale the maintenance "
        "team can understand and challenge. If Safety & Governance returns "
        "REJECTED, stop and notify the plant manager. If any specialist input "
        "is missing or based on stale data, flag it and defer rather than "
        "scoring with incomplete information."
    ),
    verbose=True,
    allow_delegation=True,      # the orchestrator delegates to the specialists
)


# 2. RELIABILITY INTELLIGENCE AGENT — asset health & failure risk
reliability_agent = Agent(
    role="Reliability Intelligence Agent",
    goal=(
        "Assess the health and failure risk of a specific asset from telemetry, "
        "sensor data and maintenance history, and return a structured "
        "assessment for the Orchestrator."
    ),
    backstory=(
        "You are the Reliability Intelligence Agent for Titan Manufacturing "
        "Corporation. Your SOLE responsibility is to assess the health and "
        "failure risk of a specific asset using telemetry, sensor readings and "
        "maintenance history. You do NOT take actions — you provide analysis.\n\n"
        "On every request you must: (1) call sensor_telemetry_api with a 72h "
        "window; (2) call anomaly_detector and interpret the severity score; "
        "(3) call maintenance_history_api for the last 10 tickets to find "
        "recurring patterns; (4) call rul_predictor to estimate remaining useful "
        "life in hours; (5) call asset_health_scorer. Do NOT call "
        "work_order_generator or notification_service — those are "
        "Orchestrator-level decisions.\n\n"
        "REASONING RULES: vibration > 0.35g on rotating equipment -> AT_RISK "
        "minimum. Temperature > 15% above baseline for > 2h -> flag anomaly. "
        "RUL < 72h AND anomaly_severity > 0.8 AND failure_probability_7d > 0.6 "
        "-> CRITICAL. Always cite the specific sensor signal driving your "
        "highest concern. If telemetry is null or stale (>15 min), flag a data "
        "quality issue and defer rather than guessing."
    ),
    tools=[sensor_telemetry_api, anomaly_detector, maintenance_history_api,
           rul_predictor, asset_health_scorer],
    verbose=True,
    allow_delegation=False,
)


# 3. PROCESS INTELLIGENCE AGENT — workflow readiness & bottlenecks
process_agent = Agent(
    role="Process Intelligence Agent",
    goal=(
        "Assess the maintenance workflow itself — parts, technicians and "
        "recurring process bottlenecks — that could slow or block a maintenance "
        "action even when the technical recommendation is correct."
    ),
    backstory=(
        "You are the Process Intelligence Agent for Titan Manufacturing "
        "Corporation. Your sole responsibility is the maintenance workflow — not "
        "the machine's health, but how efficiently the organization responds "
        "once a risk is identified. You surface bottlenecks, delays and "
        "resourcing gaps. You do NOT take actions — you provide analysis.\n\n"
        "On every request you must: (1) call maintenance_history_api (last 10 "
        "tickets) to see how similar past issues were handled and how long "
        "resolution took; (2) call parts_inventory_api to check spare-parts "
        "availability and lead time; (3) check technician availability and "
        "workload for the relevant plant and shift; (4) call root_cause_analyzer "
        "to see whether this matches a recurring process-level pattern (e.g. "
        "repeated procurement delays, technician reassignment); (5) identify the "
        "single largest bottleneck risk to timely resolution. Do NOT call "
        "work_order_generator or notification_service.\n\n"
        "REASONING RULES: if parts are unavailable and lead time exceeds the "
        "asset's RUL, classify process_risk HIGH regardless of other factors and "
        "flag it prominently — it can make a technically correct recommendation "
        "impossible to execute in time. If no technician is available on the "
        "current or next shift, treat it as a bottleneck even if parts are "
        "ready. If the asset has a documented history of repeated process "
        "delays, state the pattern explicitly so the Orchestrator can factor it "
        "into the rationale, not just the numeric score."
    ),
    tools=[maintenance_history_api, parts_inventory_api, root_cause_analyzer],
    verbose=True,
    allow_delegation=False,
)


# 4. COST INTELLIGENCE AGENT — financial & business impact
cost_agent = Agent(
    role="Cost Intelligence Agent",
    goal=(
        "Quantify the business and financial impact of a potential asset "
        "failure — money, production loss and delivery impact — so the "
        "Orchestrator can prioritize correctly."
    ),
    backstory=(
        "You are the Cost Intelligence Agent for Titan Manufacturing "
        "Corporation. You translate reliability risk into money, production loss "
        "and delivery impact.\n\n"
        "On every request: (1) call downtime_cost_calculator for the daily cost "
        "of this asset going offline; (2) check the production schedule for "
        "orders due within 14 days that depend on this asset; (3) estimate "
        "cost_of_failure_now = rul_hours / 24 x daily_downtime_cost; (4) estimate "
        "cost_of_planned_maintenance = parts_cost + technician_hours x rate; "
        "(5) compute roi = cost_of_failure_now - cost_of_planned_maintenance; "
        "(6) assign a financial impact score (0-1) by daily-cost tier: "
        "> $150,000/day -> 1.0 | $50,000-$150,000 -> 0.7 | "
        "$10,000-$50,000 -> 0.4 | < $10,000 -> 0.2.\n\n"
        "You may use parts_inventory_api to estimate parts cost. You do NOT take "
        "actions — you provide analysis for the Orchestrator."
    ),
    tools=[downtime_cost_calculator, parts_inventory_api],
    verbose=True,
    allow_delegation=False,
)


# 5. SAFETY & GOVERNANCE AGENT — the final gate before any action
governance_agent = Agent(
    role="Safety & Governance Agent",
    goal=(
        "Act as the final gate before any action is executed: ensure every "
        "proposed maintenance action complies with safety procedures, financial "
        "policies and operational rules."
    ),
    backstory=(
        "You are the Safety & Governance Agent for Titan Manufacturing "
        "Corporation. You are the FINAL GATE before any action is executed. You "
        "do not assess machine health — you assess whether a proposed action is "
        "safe and authorized.\n\n"
        "Review checklist on every proposed action: (1) call policy_validator "
        "against the rules; (2) machine shutdown -> REQUIRES_APPROVAL (plant "
        "manager sign-off); (3) repair cost estimate > $10,000 -> "
        "REQUIRES_APPROVAL (finance sign-off); (4) overriding a LOTO "
        "(Lockout/Tagout) procedure -> REJECTED; (5) alert / notify / draft work "
        "order / update dashboard -> APPROVED (autonomous); (6) vendor "
        "intervention on a critical asset -> REQUIRES_APPROVAL.\n\n"
        "Return a clear decision (APPROVED / REQUIRES_APPROVAL / REJECTED), the "
        "reason, the approver required (if any), the policy references checked, "
        "and any risk flags the approver should be aware of."
    ),
    tools=[policy_validator, notification_service],
    verbose=True,
    allow_delegation=False,
)

## Cell 5 — Define the work (the maintenance event) and assemble the crew
We use CrewAI's **hierarchical** process: the Orchestrator is the `manager_agent` that receives the event and delegates to the specialist agents — exactly the *Hierarchical Multi-Agent (ReAct)* architecture in the design document. We give the crew **one** high-level task; the manager decides who does what.

The trigger event is the **CNC-Lathe-07** scenario from the design document's *Example Run*.

In [ ]:
# The trigger event from the design document's Example Run (CNC-Lathe-07).
TRIGGER_EVENT = {
    "event_type": "SENSOR_THRESHOLD_BREACH",
    "asset_id": "CNC-Lathe-07",
    "plant": "Plant 3 - Valencia",
    "signal": "spindle_vibration",
    "value": 0.48,
    "threshold": 0.35,
    "timestamp": "2026-06-20T07:14:33Z",
}

maintenance_event_task = Task(
    description=(
        "A maintenance event has been received:\n"
        f"{json.dumps(TRIGGER_EVENT, indent=2)}\n\n"
        "As the Maintenance Intelligence Orchestrator, handle this event "
        "end-to-end:\n"
        "1. Delegate to the Reliability Intelligence Agent for an asset health "
        "and failure-risk assessment.\n"
        "2. Delegate to the Process Intelligence Agent for a workflow / "
        "bottleneck assessment.\n"
        "3. Delegate to the Cost Intelligence Agent for a financial-impact "
        "assessment.\n"
        "4. Check for CONFLICTS between the three assessments and state them "
        "explicitly.\n"
        "5. Compute Priority = (failure_probability x 0.4) + "
        "(financial_impact_score x 0.35) + (process_risk x 0.25).\n"
        "6. Propose an action and delegate it to the Safety & Governance Agent "
        "for approval.\n"
        "7. Decide AUTONOMOUS vs ESCALATE per the hard constraints (a CNC "
        "machine may never be taken offline autonomously; repairs > $10,000 "
        "always escalate; a Safety REJECTED stops everything)."
    ),
    expected_output=(
        "A single JSON object with these fields:\n"
        "{\n"
        '  "asset_id": string,\n'
        '  "plant": string,\n'
        '  "priority_score": number (0-1),\n'
        '  "risk_level": "LOW" | "MEDIUM" | "HIGH" | "CRITICAL",\n'
        '  "conflicts_detected": string (\"\" if none),\n'
        '  "recommended_action": string,\n'
        '  "autonomous_or_escalate": "AUTONOMOUS" | "ESCALATE",\n'
        '  "rationale": string (2-3 sentences, including how any conflict '
        'was resolved),\n'
        '  "work_order_draft": object or null,\n'
        '  "next_check_in": ISO-8601 timestamp\n'
        "}"
    ),
    agent=orchestrator,
)

# Hierarchical crew: manager_agent runs the show, specialists are the workers.
crew = Crew(
    agents=[reliability_agent, process_agent, cost_agent, governance_agent],
    tasks=[maintenance_event_task],
    process=Process.hierarchical,
    manager_agent=orchestrator,
    verbose=True,
)

## Cell 6 — Run the crew
Colab-safe kickoff (the same `run_crew_in_thread` helper from Module 4 of the lab), so it works inside Colab's already-running event loop.

In [ ]:
def run_crew_in_thread(crew):
    """Runs the crew on its own event loop so it works inside Colab/Jupyter,
    which already have a running loop. (Straight from the lab guide.)"""
    import threading
    result_container = {}

    def _run():
        import asyncio
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            result_container["output"] = loop.run_until_complete(crew.kickoff_async())
        finally:
            loop.close()

    t = threading.Thread(target=_run)
    t.start()
    t.join()
    return result_container["output"]

# --- Run it (notebook version, no __main__ guard needed) ---
print(f"\nHandling maintenance event for: {TRIGGER_EVENT['asset_id']} "
      f"@ {TRIGGER_EVENT['plant']}\n")

result = run_crew_in_thread(crew)

print("\n" + "=" * 64)
print("TITAN MIP — ORCHESTRATOR DECISION")
print("=" * 64)
print(result.raw)

## ✅ What just happened
The Orchestrator received the sensor alert, delegated to the **Reliability**, **Process** and **Cost** specialists, checked their assessments for **conflicts**, computed the priority score, passed the proposed action through the **Safety & Governance** gate, and returned a single structured decision (**AUTONOMOUS** or **ESCALATE**) with a rationale a technician can challenge.

**Try it yourself:** change the `TRIGGER_EVENT` in Cell 5 (e.g. a different `asset_id`, a lower `value` below the 0.35 threshold) and re-run Cells 5–6 to see the LOW-risk / autonomous path.